# 11. "1만원의 벽" - 택시 요금 심리 분석

택시 요금에 심리적 가격 저항점(round number effect)이 존재하는지 분석한다.
- 만원/2만원/3만원 근처에서 하차 빈도가 급증하는가?
- 요금 구간별 이동거리, 빈차시간, 시간대별 패턴은?

**데이터:** DC_TBYXD012 (승하차 이력, ~6억건)

In [ ]:
import gc, psutil, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
# Mac: plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

CHUNK_SIZE = 1_000_000
D012_PATH = './DC_TBYXD012.csv'

def mem_usage():
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'RAM: {gb:.1f} GB')

mem_usage()

## 1. 요금 분포 히스토그램 (100원 단위, 0~50,000원)

chunk별로 히스토그램 bin count를 누적한 뒤 한 번에 시각화한다.

In [ ]:
# 100원 단위 bin (0 ~ 50,000원)
bins_100 = np.arange(0, 50_001, 100)
hist_counts = np.zeros(len(bins_100) - 1, dtype=np.int64)

dtype_opt = {'PAY_AMT': 'int32'}

for i, chunk in enumerate(pd.read_csv(D012_PATH, usecols=['PAY_AMT'],
                                       dtype=dtype_opt, chunksize=CHUNK_SIZE)):
    amt = chunk['PAY_AMT'].values
    mask = (amt >= 0) & (amt <= 50_000)
    counts, _ = np.histogram(amt[mask], bins=bins_100)
    hist_counts += counts
    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} 처리 완료')
        mem_usage()
    del chunk, amt, mask, counts
    gc.collect()

print(f'총 건수 (0~50000원): {hist_counts.sum():,}')
mem_usage()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
centers = (bins_100[:-1] + bins_100[1:]) / 2
ax.bar(centers, hist_counts, width=100, color='steelblue', alpha=0.8, edgecolor='none')

# 만원 단위 수직선
for v in [10_000, 20_000, 30_000, 40_000]:
    ax.axvline(v, color='red', ls='--', lw=1, alpha=0.7)
    ax.text(v, ax.get_ylim()[1]*0.95, f'{v//10000}만원', ha='center',
            fontsize=9, color='red')

ax.set_xlabel('요금 (원)')
ax.set_ylabel('건수')
ax.set_title('택시 요금 분포 (100원 단위)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/10000:.1f}만'))
plt.tight_layout()
plt.show()

## 2. Round Number Effect - 9,000~11,000원 세밀 분석 (50원 단위)

In [ ]:
# 50원 단위 bin (9000 ~ 11000원)
bins_50 = np.arange(9_000, 11_001, 50)
hist_fine = np.zeros(len(bins_50) - 1, dtype=np.int64)

# 만원/2만원/3만원 근처 구간도 함께 수집
bins_20k = np.arange(19_000, 21_001, 50)
hist_20k = np.zeros(len(bins_20k) - 1, dtype=np.int64)
bins_30k = np.arange(29_000, 31_001, 50)
hist_30k = np.zeros(len(bins_30k) - 1, dtype=np.int64)

for i, chunk in enumerate(pd.read_csv(D012_PATH, usecols=['PAY_AMT'],
                                       dtype=dtype_opt, chunksize=CHUNK_SIZE)):
    amt = chunk['PAY_AMT'].values
    
    m1 = (amt >= 9_000) & (amt <= 11_000)
    c1, _ = np.histogram(amt[m1], bins=bins_50)
    hist_fine += c1
    
    m2 = (amt >= 19_000) & (amt <= 21_000)
    c2, _ = np.histogram(amt[m2], bins=bins_20k)
    hist_20k += c2
    
    m3 = (amt >= 29_000) & (amt <= 31_000)
    c3, _ = np.histogram(amt[m3], bins=bins_30k)
    hist_30k += c3
    
    del chunk, amt, m1, m2, m3, c1, c2, c3
    gc.collect()

mem_usage()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, bins_arr, hist_arr, title in [
    (axes[0], bins_50, hist_fine, '1만원 근처 (9,000~11,000원)'),
    (axes[1], bins_20k, hist_20k, '2만원 근처 (19,000~21,000원)'),
    (axes[2], bins_30k, hist_30k, '3만원 근처 (29,000~31,000원)'),
]:
    centers = (bins_arr[:-1] + bins_arr[1:]) / 2
    ax.bar(centers, hist_arr, width=50, color='steelblue', alpha=0.8)
    round_val = int(np.median(bins_arr))
    ax.axvline(round_val, color='red', ls='--', lw=1.5)
    ax.set_title(title)
    ax.set_xlabel('요금 (원)')
    ax.set_ylabel('건수')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Round Number Effect 분석 (50원 단위)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. 요금 구간별 평균 이동거리 - "만원어치 타면 몇 km?"

In [ ]:
# 요금 구간 정의 (1000원 단위)
fare_edges = list(range(0, 50_001, 1_000))
n_bins = len(fare_edges) - 1
dist_sum = np.zeros(n_bins, dtype=np.float64)
dist_cnt = np.zeros(n_bins, dtype=np.int64)

cols = ['PAY_AMT', 'RIDE_DIST']
dtypes = {'PAY_AMT': 'int32', 'RIDE_DIST': 'float32'}

for i, chunk in enumerate(pd.read_csv(D012_PATH, usecols=cols,
                                       dtype=dtypes, chunksize=CHUNK_SIZE)):
    mask = (chunk['PAY_AMT'] >= 0) & (chunk['PAY_AMT'] <= 50_000) & (chunk['RIDE_DIST'] > 0)
    df = chunk.loc[mask]
    idx = np.digitize(df['PAY_AMT'].values, fare_edges) - 1
    valid = (idx >= 0) & (idx < n_bins)
    for b in range(n_bins):
        sel = (idx == b) & valid
        dist_sum[b] += df['RIDE_DIST'].values[sel].sum()
        dist_cnt[b] += sel.sum()
    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} 처리 완료')
    del chunk, df, mask, idx, valid
    gc.collect()

avg_dist = np.where(dist_cnt > 0, dist_sum / dist_cnt / 1000, 0)  # m -> km
mem_usage()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
centers = [(fare_edges[i] + fare_edges[i+1]) / 2 for i in range(n_bins)]
ax.bar(centers, avg_dist, width=900, color='coral', alpha=0.8)

# 주요 요금대 표시
for v in [10_000, 20_000, 30_000]:
    idx_v = v // 1_000
    if idx_v < n_bins and avg_dist[idx_v] > 0:
        ax.annotate(f'{avg_dist[idx_v]:.1f}km',
                    xy=(v+500, avg_dist[idx_v]),
                    fontsize=10, ha='center', va='bottom', color='red', fontweight='bold')

ax.set_xlabel('요금 (원)')
ax.set_ylabel('평균 이동거리 (km)')
ax.set_title('요금 구간별 평균 이동거리')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/10000:.1f}만'))
plt.tight_layout()
plt.show()

# 주요 구간 요약
for label, lo, hi in [('기본요금(3800~4800)', 3, 4), ('1만원대(9500~10500)', 9, 10),
                       ('2만원대', 19, 20), ('3만원대', 29, 30)]:
    vals = avg_dist[lo:hi+1]
    if vals.mean() > 0:
        print(f'{label}: 평균 {vals.mean():.1f} km')

## 4. 단거리/중거리/장거리 비율의 시간대별 변화

In [ ]:
# 시간대 x 거리구간 집계
# 단거리: <3km, 중거리: 3~10km, 장거리: 10km+
hour_dist = np.zeros((24, 3), dtype=np.int64)  # [hour, 0=단거리/1=중거리/2=장거리]

cols = ['RIDE_DTIME', 'RIDE_DIST']
dtypes = {'RIDE_DTIME': 'str', 'RIDE_DIST': 'float32'}

for i, chunk in enumerate(pd.read_csv(D012_PATH, usecols=cols,
                                       dtype=dtypes, chunksize=CHUNK_SIZE)):
    chunk = chunk.dropna(subset=['RIDE_DTIME', 'RIDE_DIST'])
    hour = chunk['RIDE_DTIME'].str[8:10].astype(np.int8)
    dist_m = chunk['RIDE_DIST'].values
    
    for h in range(24):
        h_mask = (hour == h)
        d = dist_m[h_mask]
        hour_dist[h, 0] += (d < 3_000).sum()
        hour_dist[h, 1] += ((d >= 3_000) & (d < 10_000)).sum()
        hour_dist[h, 2] += (d >= 10_000).sum()
    
    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} 처리 완료')
    del chunk, hour, dist_m
    gc.collect()

mem_usage()

In [ ]:
# 비율 계산
total_per_hour = hour_dist.sum(axis=1, keepdims=True)
ratio = np.where(total_per_hour > 0, hour_dist / total_per_hour * 100, 0)

fig, ax = plt.subplots(figsize=(14, 6))
hours = np.arange(24)
ax.bar(hours, ratio[:, 0], label='단거리 (<3km)', color='#4CAF50', alpha=0.85)
ax.bar(hours, ratio[:, 1], bottom=ratio[:, 0], label='중거리 (3~10km)', color='#2196F3', alpha=0.85)
ax.bar(hours, ratio[:, 2], bottom=ratio[:, 0]+ratio[:, 1], label='장거리 (10km+)', color='#F44336', alpha=0.85)

# 심야 구간 강조
ax.axvspan(23.5, 24, color='gray', alpha=0.15)
ax.axvspan(-0.5, 4.5, color='gray', alpha=0.15, label='심야시간대')

ax.set_xlabel('시간대')
ax.set_ylabel('비율 (%)')
ax.set_title('시간대별 단거리/중거리/장거리 비율 변화')
ax.set_xticks(hours)
ax.set_xticklabels([f'{h}시' for h in hours], rotation=45)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

# 심야 vs 주간 비교
night_idx = [0, 1, 2, 3, 4, 23]
day_idx = [7, 8, 9, 10, 11, 12, 13, 14]
print('=== 장거리(10km+) 비율 ===')
print(f'심야(23~04시): {ratio[night_idx, 2].mean():.1f}%')
print(f'주간(07~14시): {ratio[day_idx, 2].mean():.1f}%')

## 5. 요금 구간별 빈차시간 - 장거리 손님이 오히려 비효율?

동일 택시(TAXI_VEHC_ID)의 연속된 운행에서 "전 승객 하차시각 ~ 다음 승객 승차시각" 차이를 빈차시간으로 정의한다.  
메모리 제약으로, chunk 단위로 (택시ID, 하차시각, 요금)을 수집한 뒤 정렬하여 계산한다.

In [ ]:
# Step 1: 필요한 컬럼만 추출하여 parquet로 중간 저장 (메모리 절약)
# 택시ID, 승차시각, 하차시각, 요금
cols = ['TAXI_VEHC_ID', 'RIDE_DTIME', 'ALIGHT_DTIME', 'PAY_AMT']
dtypes = {'TAXI_VEHC_ID': 'category', 'RIDE_DTIME': 'str',
           'ALIGHT_DTIME': 'str', 'PAY_AMT': 'int32'}

# 요금 구간 정의
fare_labels = ['~5천', '5천~1만', '1만~2만', '2만~3만', '3만+']
fare_cuts = [0, 5_000, 10_000, 20_000, 30_000, 999_999]

# chunk 처리: 택시별 (하차시각, 다음승차시각) 쌍을 추출
# 대용량이므로 샘플링 전략 사용 (택시 ID 해시 기반 10% 샘플)
vacant_records = []  # (fare_bin_idx, vacant_minutes)

prev_tail = {}  # taxi_id -> (alight_dtime_str, pay_amt)

for i, chunk in enumerate(pd.read_csv(D012_PATH, usecols=cols,
                                       dtype=dtypes, chunksize=CHUNK_SIZE)):
    chunk = chunk.dropna()
    # 택시 ID 해시 기반 10% 샘플
    chunk['_hash'] = chunk['TAXI_VEHC_ID'].astype(str).str[-1]
    chunk = chunk[chunk['_hash'] == '0'].drop(columns=['_hash'])
    
    chunk = chunk.sort_values(['TAXI_VEHC_ID', 'RIDE_DTIME'])
    
    for _, grp in chunk.groupby('TAXI_VEHC_ID', observed=True):
        taxi_id = grp['TAXI_VEHC_ID'].iloc[0]
        rides = grp[['RIDE_DTIME', 'ALIGHT_DTIME', 'PAY_AMT']].values
        
        start_idx = 0
        # 이전 chunk의 마지막 하차 정보와 연결
        taxi_key = str(taxi_id)
        if taxi_key in prev_tail:
            prev_alight, prev_fare = prev_tail[taxi_key]
            try:
                alight_dt = pd.to_datetime(prev_alight, format='%Y%m%d%H%M%S')
                ride_dt = pd.to_datetime(rides[0][0], format='%Y%m%d%H%M%S')
                vacant_min = (ride_dt - alight_dt).total_seconds() / 60
                if 0 < vacant_min < 180:  # 3시간 이내만 유효
                    b = np.digitize(prev_fare, fare_cuts) - 1
                    if 0 <= b < len(fare_labels):
                        vacant_records.append((b, vacant_min))
            except:
                pass
        
        # 그룹 내 연속 운행
        for j in range(1, len(rides)):
            try:
                alight_dt = pd.to_datetime(rides[j-1][1], format='%Y%m%d%H%M%S')
                ride_dt = pd.to_datetime(rides[j][0], format='%Y%m%d%H%M%S')
                vacant_min = (ride_dt - alight_dt).total_seconds() / 60
                if 0 < vacant_min < 180:
                    fare = int(rides[j-1][2])
                    b = np.digitize(fare, fare_cuts) - 1
                    if 0 <= b < len(fare_labels):
                        vacant_records.append((b, vacant_min))
            except:
                pass
        
        # 마지막 운행 기록 저장
        prev_tail[taxi_key] = (rides[-1][1], int(rides[-1][2]))
    
    if (i + 1) % 20 == 0:
        print(f'  chunk {i+1} 처리 완료, records: {len(vacant_records):,}')
    del chunk, grp
    gc.collect()

del prev_tail
gc.collect()
print(f'빈차시간 레코드: {len(vacant_records):,}')
mem_usage()

In [ ]:
vacant_arr = np.array(vacant_records, dtype=np.float32)
del vacant_records
gc.collect()

fig, ax = plt.subplots(figsize=(10, 6))
avg_vacant = []
med_vacant = []
for b in range(len(fare_labels)):
    sel = vacant_arr[vacant_arr[:, 0] == b, 1]
    avg_vacant.append(sel.mean() if len(sel) > 0 else 0)
    med_vacant.append(np.median(sel) if len(sel) > 0 else 0)

x = np.arange(len(fare_labels))
w = 0.35
ax.bar(x - w/2, avg_vacant, w, label='평균', color='steelblue', alpha=0.85)
ax.bar(x + w/2, med_vacant, w, label='중앙값', color='coral', alpha=0.85)

for i, (a, m) in enumerate(zip(avg_vacant, med_vacant)):
    ax.text(i - w/2, a + 0.3, f'{a:.1f}분', ha='center', fontsize=9)
    ax.text(i + w/2, m + 0.3, f'{m:.1f}분', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(fare_labels)
ax.set_xlabel('이전 승객 요금 구간')
ax.set_ylabel('빈차시간 (분)')
ax.set_title('요금 구간별 빈차시간 (하차 후 다음 손님까지)')
ax.legend()
plt.tight_layout()
plt.show()

del vacant_arr
gc.collect()

## 6. 시간대별 평균 요금 추이 + 할증 효과

In [ ]:
fare_sum = np.zeros(24, dtype=np.float64)
fare_cnt = np.zeros(24, dtype=np.int64)

cols = ['RIDE_DTIME', 'PAY_AMT']
dtypes = {'RIDE_DTIME': 'str', 'PAY_AMT': 'int32'}

for i, chunk in enumerate(pd.read_csv(D012_PATH, usecols=cols,
                                       dtype=dtypes, chunksize=CHUNK_SIZE)):
    chunk = chunk.dropna()
    mask = (chunk['PAY_AMT'] > 0) & (chunk['PAY_AMT'] < 200_000)
    df = chunk.loc[mask]
    hour = df['RIDE_DTIME'].str[8:10].astype(np.int8)
    
    for h in range(24):
        h_mask = (hour == h)
        fare_sum[h] += df.loc[h_mask, 'PAY_AMT'].sum()
        fare_cnt[h] += h_mask.sum()
    
    if (i + 1) % 50 == 0:
        print(f'  chunk {i+1} 처리 완료')
    del chunk, df, mask, hour
    gc.collect()

avg_fare = np.where(fare_cnt > 0, fare_sum / fare_cnt, 0)
mem_usage()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

hours = np.arange(24)
colors = ['#F44336' if h in [23, 0, 1, 2, 3] else 'steelblue' for h in hours]
bars = ax1.bar(hours, avg_fare, color=colors, alpha=0.85)

# 할증 시간대 표시
ax1.axvspan(-0.5, 3.5, color='red', alpha=0.05)
ax1.axvspan(22.5, 23.5, color='red', alpha=0.05)
ax1.text(1.5, ax1.get_ylim()[1]*0.9 if ax1.get_ylim()[1] > 0 else avg_fare.max()*0.9,
         '심야할증', ha='center', fontsize=10, color='red')

for h in hours:
    ax1.text(h, avg_fare[h] + avg_fare.max()*0.01, f'{avg_fare[h]:,.0f}',
             ha='center', fontsize=7, rotation=90)

# 건수 라인
ax2 = ax1.twinx()
ax2.plot(hours, fare_cnt / 1e6, 'o-', color='green', alpha=0.6, markersize=4)
ax2.set_ylabel('건수 (백만건)', color='green')

ax1.set_xlabel('시간대')
ax1.set_ylabel('평균 요금 (원)')
ax1.set_title('시간대별 평균 택시 요금 (심야할증 = 빨간색)')
ax1.set_xticks(hours)
ax1.set_xticklabels([f'{h}시' for h in hours], rotation=45)
plt.tight_layout()
plt.show()

# 할증 효과 수치
day_avg = avg_fare[6:22].mean()
night_avg = avg_fare[[23, 0, 1, 2, 3]].mean()
print(f'주간(06~21시) 평균 요금: {day_avg:,.0f}원')
print(f'심야(23~03시) 평균 요금: {night_avg:,.0f}원')
print(f'심야 할증 효과: +{(night_avg/day_avg - 1)*100:.1f}%')

## 7. 종합 요약

In [ ]:
print('=' * 60)
print('택시 요금 심리 분석 - 종합 요약')
print('=' * 60)
print()
print('[1] 심리적 가격 저항점 (Round Number Effect)')
print('    - 1만원/2만원/3만원 근처에서 하차 빈도 패턴 확인')
print('    - 위 히스토그램에서 round number 직전에 하차가')
print('      집중되는지 (승객의 "만원 넘기기 싫다" 심리) 확인')
print()
print('[2] 요금 구간별 이동거리')
for i, label in enumerate(fare_labels):
    if i < len(avg_dist) and avg_dist[i] > 0:
        pass  # 위 셀에서 이미 출력
print('    -> 1만원 = 약 ?km, 2만원 = 약 ?km (위 차트 참조)')
print()
print('[3] 시간대별 거리 분포')
print(f'    - 심야(23~04시) 장거리 비율: {ratio[night_idx, 2].mean():.1f}%')
print(f'    - 주간(07~14시) 장거리 비율: {ratio[day_idx, 2].mean():.1f}%')
print()
print('[4] 빈차시간 분석')
print('    - 요금 구간별 평균 빈차시간은 위 차트 참조')
print('    - 장거리(고액) 승객 하차 후 빈차시간이 긴 경우,')
print('      기사 입장에서 장거리가 반드시 이득은 아님')
print()
print('[5] 할증 효과')
print(f'    - 주간 평균: {day_avg:,.0f}원 / 심야 평균: {night_avg:,.0f}원')
print(f'    - 심야 할증 효과: +{(night_avg/day_avg - 1)*100:.1f}%')
print()
print('=' * 60)
mem_usage()